# SVM Baseline with Class Weighting and TF-IDF Tuning
This notebook trains a strong classical baseline for the PROMISE expanded requirements dataset.

For SVM, we avoid text augmentation as the main setting because sparse TF-IDF features are sensitive to synthetic lexical noise. Instead, we use class weighting and select the best TF-IDF/SVM configuration on the validation set before evaluating once on the test set.

In [ ]:
!pip install transformers pandas torch scikit-learn
!pip install nltk spacy word2number
!python -m spacy download en_core_web_sm
!pip install -U transformers

In [ ]:
# --- Imports --- 
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt
import seaborn as sns
import random
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import random
import re
import string
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer
import nltk
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.pipeline import Pipeline, FeatureUnion


In [ ]:
from transformers.models.vision_text_dual_encoder import processing_vision_text_dual_encoder
# --- Setup ---
os.environ["WANDB_DISABLED"] = "true"
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# --- Text Processing ---
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = [word for word in text.split() if word not in stop_words]
    text = ' '.join(stemmer.stem(word) for word in tokens)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != word:
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, n=1):
    words = text.split()
    new_words = words.copy()
    random_word_list = list(set([word for word in words if word not in stop_words and len(word) > 3]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(synonyms)
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)

def phrase_insertion(text, n=1):
    phrases = ["the system shall", "must be able to", "should ensure that"]
    words = text.split()
    for _ in range(n):
        add_phrase = random.choice(phrases)
        random_idx = random.randint(0, len(words)-1)
        words.insert(random_idx, add_phrase)
    return ' '.join(words)

def text_simplification(text):
    text = re.sub(r'\bthe system shall\b', '', text)
    text = re.sub(r'\bmust be able to\b', '', text)
    text = re.sub(r'\bshould ensure that\b', '', text)
    return text.strip()

def random_deletion(text, p=0.1):
    words = text.split()
    if len(words) == 1:
        return text
    remaining = [word for word in words if random.random() > p]
    return ' '.join(remaining) if remaining else random.choice(words)

def random_swap(text, n=1):
    words = text.split()
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    return ' '.join(words)

## --- Load Data ---
DATA_PATH_CANDIDATES = [
    os.path.join('..', 'data', 'exp', 'promise_exp.csv'),
    os.path.join('data', 'exp', 'promise_exp.csv'),
    r'D:\semester4\SWR302\RBL_requirements_classification\rbl-requirements-classification\data\exp\promise_exp.csv',
]

for data_path in DATA_PATH_CANDIDATES:
    if os.path.exists(data_path):
        break
else:
    raise FileNotFoundError(
        "Could not find promise_exp.csv. Expected it under data/exp/promise_exp.csv."
    )

df = pd.read_csv(data_path)
df = df.rename(columns={'RequirementText': 'text', 'class': 'label'})
df = df[['text', 'label']].dropna().reset_index(drop=True)
df = df[df['label'].map(df['label'].value_counts()) > 1].reset_index(drop=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['label'],
    random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Loaded PROMISE expanded dataset from: {data_path}")
print("Train Distribution:", dict(Counter(train_df['label'])))
print("Validation Distribution:", dict(Counter(val_df['label'])))
print("Test Distribution:", dict(Counter(test_df['label'])))

# --- Preprocessing ---
train_df['cleaned_text'] = train_df['text'].apply(lambda x: clean_text(str(x)))
val_df['cleaned_text'] = val_df['text'].apply(lambda x: clean_text(str(x)))
test_df['cleaned_text'] = test_df['text'].apply(lambda x: clean_text(str(x)))
df_combined = train_df.copy()

print("SVM baseline uses real training samples only with class_weight='balanced'.")
print("Training Distribution:", dict(Counter(df_combined['label'])))
print("\n")


In [ ]:

## ---- Strong SVM Baseline: TF-IDF + Class Weighting + Validation Selection ---
from IPython.display import display, HTML
from sklearn.metrics import f1_score

y_train = df_combined['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

text_sets = {
    'paper_preprocessed': (
        df_combined['cleaned_text'].astype(str),
        val_df['cleaned_text'].astype(str),
        test_df['cleaned_text'].astype(str),
    ),
    'raw_text': (
        df_combined['text'].astype(str),
        val_df['text'].astype(str),
        test_df['text'].astype(str),
    ),
}

feature_configs = {
    'word_1_2': TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=20000, sublinear_tf=True, min_df=1, max_df=0.95, norm='l2'),
    'word_1_3': TfidfVectorizer(lowercase=True, ngram_range=(1, 3), max_features=30000, sublinear_tf=True, min_df=1, max_df=0.95, norm='l2'),
    'word_char': FeatureUnion([
        ('word', TfidfVectorizer(lowercase=True, analyzer='word', ngram_range=(1, 2), max_features=20000, sublinear_tf=True, min_df=1, max_df=0.95, norm='l2')),
        ('char', TfidfVectorizer(lowercase=True, analyzer='char_wb', ngram_range=(3, 5), max_features=20000, sublinear_tf=True, min_df=1, norm='l2')),
    ]),
}

c_values = [0.25, 0.5, 1.0, 2.0, 4.0]
results = []
best = None

start_time = time.time()
for text_name, (X_train_text, X_val_text, X_test_text) in text_sets.items():
    for feature_name, features in feature_configs.items():
        for c in c_values:
            model = Pipeline([
                ('features', features),
                ('classifier', LinearSVC(C=c, class_weight='balanced', random_state=RANDOM_STATE, max_iter=20000, dual='auto')),
            ])
            model.fit(X_train_text, y_train)
            y_val_pred = model.predict(X_val_text)
            row = {
                'text': text_name,
                'features': feature_name,
                'C': c,
                'val_accuracy': accuracy_score(y_val, y_val_pred),
                'val_macro_f1': f1_score(y_val, y_val_pred, average='macro', zero_division=0),
                'val_weighted_f1': f1_score(y_val, y_val_pred, average='weighted', zero_division=0),
                'model': model,
                'X_test_text': X_test_text,
            }
            results.append(row)
            if best is None or row['val_macro_f1'] > best['val_macro_f1']:
                best = row

train_time = time.time() - start_time
results_df = pd.DataFrame([{k: v for k, v in row.items() if k not in ['model', 'X_test_text']} for row in results])
results_df = results_df.sort_values(['val_macro_f1', 'val_weighted_f1', 'val_accuracy'], ascending=False).reset_index(drop=True)
display(HTML(results_df.to_html(index=False)))

print(f"SVM validation search time: {train_time:.4f} seconds")
print("\nBest validation configuration:")
print({k: best[k] for k in ['text', 'features', 'C', 'val_accuracy', 'val_macro_f1', 'val_weighted_f1']})

best_model = best['model']
y_test_pred = best_model.predict(best['X_test_text'])

print("\nTest metrics for selected SVM baseline:")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Macro F1: {f1_score(y_test, y_test_pred, average='macro', zero_division=0):.4f}")
print(f"Weighted F1: {f1_score(y_test, y_test_pred, average='weighted', zero_division=0):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

labels = np.unique(y_test)
cm_multi = confusion_matrix(y_test, y_test_pred, labels=labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Purples', xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix - SVM Baseline (Validation-Selected)")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()